# 실습 07 · 예지보수: 설비 잔여수명(RUL) 예측
### 센서로 고장을 미리 예측하는 Predictive Maintenance

**왜 중요한가 (현장 맥락)**
제약 설비(펌프·압축기·HVAC·동결건조기)가 갑자기 멈추면 **배치 전체를 폐기**할 수 있습니다.
정기점검(시간 기반)은 낭비가 크고, 고장 후 수리는 위험합니다.
**센서 데이터로 남은 수명(RUL, Remaining Useful Life)을 예측**해
'딱 필요할 때' 정비하는 것이 예지보수입니다.

> 데이터는 NASA **C-MAPSS 터보팬** 열화 데이터를 본뜬 현실적 시뮬레이션입니다.
> (제약 설비도 동일한 방법론이 그대로 적용됩니다)


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(7)

# ---- 여러 설비(unit)의 열화 이력 생성 ----
# 각 설비는 정상 → 점진적 열화 → 고장. 센서 3종이 열화에 따라 변함.
def make_unit(uid, life):
    t = np.arange(1, life+1)
    deg = (t/life)                       # 0→1 로 열화 진행
    s_vib  = 0.5 + 2.0*deg**2 + np.random.normal(0,0.05,life)  # 진동↑
    s_temp = 60 + 15*deg   + np.random.normal(0,0.5,life)      # 온도↑
    s_cur  = 10 + 4*deg**1.5+ np.random.normal(0,0.1,life)     # 전류↑
    rul = life - t                       # 잔여수명(정답)
    return pd.DataFrame({"unit":uid,"cycle":t,"진동":s_vib,
                         "온도":s_temp,"전류":s_cur,"RUL":rul})

lives = np.random.randint(120, 220, 80)   # 80대 설비, 수명 제각각
df = pd.concat([make_unit(i, L) for i,L in enumerate(lives)], ignore_index=True)
print("총 관측(설비×사이클):", len(df))
df.head()

In [ ]:
# 한 설비의 센서가 고장에 가까워지며 어떻게 변하는지 시각화
u = df[df.unit==0]
fig, ax = plt.subplots(1,3, figsize=(13,3.2))
for a,c in zip(ax, ["진동","온도","전류"]):
    a.plot(u["cycle"], u[c]); a.set_title(c); a.set_xlabel("cycle")
plt.suptitle("설비 #0: 시간이 갈수록 센서값이 열화 신호를 보임"); plt.tight_layout(); plt.show()

## 1. RUL 회귀 예측
현재 센서값(+최근 추세)으로 **남은 수명(RUL)** 을 예측합니다.
현장 팁: 순간값뿐 아니라 **이동평균/기울기** 같은 추세 특성을 넣으면 성능이 크게 오릅니다.


In [ ]:
# 추세 특성 추가 (설비별 롤링 평균) — 실무에서 매우 중요한 단계
for c in ["진동","온도","전류"]:
    df[c+"_ma"] = df.groupby("unit")[c].transform(lambda s: s.rolling(5,min_periods=1).mean())

feats = ["진동","온도","전류","진동_ma","온도_ma","전류_ma","cycle"]

# 설비 단위로 학습/평가 분리 (같은 설비가 양쪽에 섞이면 안 됨 = 데이터 누수 방지)
units = df.unit.unique(); np.random.shuffle(units)
train_u = set(units[:60]); 
tr = df[df.unit.isin(train_u)]; te = df[~df.unit.isin(train_u)]

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
reg = RandomForestRegressor(n_estimators=300, random_state=42).fit(tr[feats], tr["RUL"])
pred = reg.predict(te[feats])
print("RUL 예측 MAE:", round(mean_absolute_error(te['RUL'],pred),1), "cycle")
print("RUL 예측 R² :", round(r2_score(te['RUL'],pred),3))

In [ ]:
# 한 평가 설비의 실제 RUL vs 예측 RUL (아래로 갈수록 고장 임박)
te = te.copy(); te["예측RUL"] = pred
one = te[te.unit==te.unit.unique()[0]].sort_values("cycle")
plt.figure(figsize=(8,4))
plt.plot(one["cycle"], one["RUL"], label="실제 RUL")
plt.plot(one["cycle"], one["예측RUL"], "--", label="예측 RUL")
plt.axhline(20, color="red", ls=":", label="정비 임계값(RUL=20)")
plt.xlabel("cycle"); plt.ylabel("잔여수명(RUL)"); plt.legend()
plt.title("설비 잔여수명 예측"); plt.show()

## 2. ⭐ 정비 의사결정 — 경보 임계값
RUL 예측을 **행동**으로 연결합니다.
예: 예측 RUL 이 20 cycle 아래로 떨어지면 **정비 작업지시(work order) 자동 발행**.


In [ ]:
THRESH = 20
te["정비경보"] = (te["예측RUL"] < THRESH).astype(int)
alerts = te[te["정비경보"]==1].groupby("unit")["cycle"].min()
print("설비별 최초 정비경보 시점(cycle):")
print(alerts)
print(f"\n평균적으로 실제 고장 {te[te.정비경보==1]['RUL'].mean():.0f} cycle 전에 경보 → 사전 정비 가능")

## 정리 & 현장 응용
- 센서값 + **추세 특성(이동평균)** 으로 잔여수명(RUL) 예측
- **설비 단위 데이터 분리**로 현실적 평가 (누수 방지)
- RUL → **정비 임계값 → 자동 작업지시** 로 실행 연결
- 제약 적용: 동결건조기, 정제수(WFI) 시스템, HVAC, 압축공기, 펌프 등
- 확장: LSTM/1D-CNN 으로 시계열 원신호 직접 학습(예제 08·다음 단계)
